### Imports

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import json
import warnings
import gc
import numpy as np
import skops.io as sio
import pandas as pd
import matplotlib.pyplot as plt
import config
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, MinMaxScaler
from sklearn.inspection import permutation_importance
from src.utils import (
    read_csv_file, 
    save_csv_file, 
    agnostic_bacc_scorer,
    compute_bacc,
    compute_logloss,
    compute_recall_per_class,
    reduce_mem_usage
)
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression, SGDClassifier
from category_encoders import CountEncoder, SumEncoder, CatBoostEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import mlflow
from tqdm.notebook import tqdm
from pytabkit import (
	RealMLP_TD_Classifier,
	TabM_D_Classifier,
	Resnet_RTDL_D_Classifier,
	CatBoost_TD_Classifier,
	LGBM_TD_Classifier,
	XGB_TD_Classifier
)
from src.fe import *
pd.set_option('display.max_columns', None)

import logging
logging.getLogger("mlflow.utils.environment").setLevel(logging.ERROR)

In [24]:
warnings.filterwarnings("ignore", category=pd.errors.ChainedAssignmentError)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

### Prepare data

In [25]:
train_raw = read_csv_file(r'data\raw\train.csv')
test_raw = read_csv_file(r'data\raw\test.csv')

# Train and test ids
train_ids = train_raw['id']
test_ids = test_raw['id']

n_classes = 3

Reading file from: data\raw\train.csv
Reading file from: data\raw\test.csv


In [26]:
# X and y split
X = train_raw.drop(['id', 'irrigation_need'], axis=1)
y = train_raw['irrigation_need'].map(config.TARGET_MAP)
classes = np.unique(y)

# Test data
test_data = test_raw.drop('id', axis=1)

In [27]:
# Reduce memory size of data
X = reduce_mem_usage(X)
test_data = reduce_mem_usage(test_data)

Memory before: 354.06 MB
Memory after: 327.63 MB
Reduced by: 26.44 MB (7.5%)
Memory before: 151.74 MB
Memory after: 140.41 MB
Reduced by: 11.33 MB (7.5%)


In [28]:
RUN = 'v1'

### Logistic Regression

In [14]:
for df in [X, test_data]:
    df["soil_lt_25"] = (df["soil_moisture"] < 25).astype(int)
    df["temp_gt_30"] = (df["temperature_c"] > 30).astype(int)
    df["rain_lt_300"] = (df["rainfall_mm"] < 300).astype(int)
    df["wind_gt_10"] = (df["wind_speed_kmh"] > 10).astype(int)
    df['mulching_used'] = df['mulching_used'].map({'Yes': 1, 'No': 0})

In [15]:
TE_columns = []

columns = config.BASE_CAT_COLS + config.BASE_NUM_COLS

for cols in tqdm(list(itertools.combinations(columns, 2))):
	name = '-'.join(cols)

	X[name] = X[cols[0]].astype(str)
	for col in cols[1:]:
		X[name] = X[name] + '_' + X[col].astype(str)

	test_data[name] = test_data[cols[0]].astype(str)
	for col in cols[1:]:
		test_data[name] = test_data[name] + '_' + test_data[col].astype(str)

	combined = pd.concat([X[name], test_data[name]], ignore_index=True)
	combined, _ = combined.factorize()
	if pd.Series(combined).nunique() > len(combined) // 2:
		X = X.drop(name, axis=1)
		test_data = test_data.drop(name, axis=1)
		continue
	X[name] = combined[:len(X)]
	test_data[name] = combined[len(X):len(X) + len(test_data)]

	TE_columns.append(name)

  0%|          | 0/171 [00:00<?, ?it/s]

In [16]:
X = X[['soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10', 'crop_growth_stage', 'mulching_used'] + TE_columns]
test_data = test_data[['soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10', 'crop_growth_stage', 'mulching_used'] + TE_columns]

In [17]:
combined = pd.concat([X, test_data])
combined = pd.get_dummies(combined, columns=['crop_growth_stage'])

In [18]:
X = combined[:len(X)]
test_data = combined[len(X):]

FEATURES = X.columns.tolist()

In [19]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [21]:
EXP_NAME = "LOGISTIC"
RUN_NAME = f"Logistic_seed{config.SEED}_{RUN}"

skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

final_oof_preds = np.zeros((len(X), n_classes))
final_test_preds = np.zeros((len(test_data), n_classes))
feature_importance = []
bacc_scores = []
ll_scores = []

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")
    mlflow.log_params({
		"seed": config.SEED,
		"n_folds": config.N_FOLDS,
		"n_cols": X.shape[1],
		**config.LOGISTIC_PARAMS
	})

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

        # -------- PIPELINE --------
        te = ColumnTransformer(
            transformers=[
                ('te', TargetEncoder(target_type='multiclass', cv=5, random_state=config.SEED), TE_columns)
			],
		remainder='passthrough').set_output(transform='pandas')

        model = Pipeline(
            steps=[
                ('encoder', te),
                ('scaler', RobustScaler()),
                ('model', LogisticRegression(**config.LOGISTIC_PARAMS))
			]
		)

        model.fit(X_train, y_train)

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
            mlflow.sklearn.log_model(model, name="model")

            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            # fi = permutation_importance(
                # model,
                # X_valid,
                # y_valid,
                # scoring='balanced_accuracy',
                # n_repeats=1,
                # n_jobs=1,
                # random_state=config.SEED
            # )
            # feature_importance.append(fi.importances_mean)

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
        "recall_0": recall_per_class[0],
        "recall_1": recall_per_class[1],
        "recall_2": recall_per_class[2]
    }

    mlflow.log_metrics(metrics)

    # -------- FEATURE IMPORTANCE --------
    #fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    #fi_df = pd.DataFrame({
    #    "feature": X_valid.columns,
    #    "importance": fi_mean
    #}).sort_values("importance", ascending=False).head(50)

    #fig, ax = plt.subplots(figsize=(10, 14))
    # ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    # ax.set_title("Top 50 Feature Importance")
    # fig.tight_layout()
    # mlflow.log_figure(fig, "feature_importance/fi.png")
    # plt.close(fig)
    
    if fold < 5:
        del model, X_train, X_valid, y_train, y_valid
        gc.collect()

# -------- SAVE FILES --------
oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)
save_csv_file(oof_df, os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv"))

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)
save_csv_file(test_df, os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv"))

2026/04/22 01:10:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 1 | Balanced Acc: 0.97587 | LogLoss: 0.06710


2026/04/22 01:21:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 2 | Balanced Acc: 0.97604 | LogLoss: 0.06844


2026/04/22 01:33:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 3 | Balanced Acc: 0.97777 | LogLoss: 0.06858


2026/04/22 01:44:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 4 | Balanced Acc: 0.97630 | LogLoss: 0.06729


2026/04/22 01:56:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 5 | Balanced Acc: 0.97639 | LogLoss: 0.06669
Saving file to: artifacts\oof\Logistic_seed42_v21_oof_proba.csv
Saving file to: artifacts\test_proba\Logistic_seed42_v21_test_proba.csv


### Histogram GBM

In [42]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [ ]:
EXP_NAME = "EXTRATREES"
RUN_NAME = f"extratrees_seed{config.SEED}_{RUN}"

skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

final_oof_preds = np.zeros((len(X), n_classes))
final_test_preds = np.zeros((len(test_data), n_classes))
feature_importance = []
bacc_scores = []
ll_scores = []

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")
    mlflow.log_params({
		"seed": config.SEED,
		"n_folds": config.N_FOLDS,
		"n_cols": X.shape[1],
		**config.EXTRATREES_PARAMS
	})

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

        # -------- PIPELINE --------
        ohe = ColumnTransformer(
            transformers=[
                ('ohe', OneHotEncoder(sparse_output=False, dtype=int), cat_cols)
			],
		remainder='passthrough').set_output(transform='pandas')

        model = Pipeline(
            steps=[
                ('ohe', ohe),
                ('model', ExtraTreesClassifier(**config.EXTRATREES_PARAMS))
			]
		)

        model.fit(X_train, y_train)

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
            mlflow.sklearn.log_model(model, name="model")

            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring='balanced_accuracy',
                n_repeats=1,
                n_jobs=-1,
                random_state=config.SEED
            )
            feature_importance.append(fi.importances_mean)

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
        "recall_0": recall_per_class[0],
        "recall_1": recall_per_class[1],
        "recall_2": recall_per_class[2]
    }

    mlflow.log_metrics(metrics)

    # -------- FEATURE IMPORTANCE --------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")
    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)
    
    if fold < 5:
        del model, X_train, X_valid, y_train, y_valid
        gc.collect()

# -------- SAVE FILES --------
oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)
save_csv_file(oof_df, os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv"))

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)
save_csv_file(test_df, os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv"))

2026/04/21 01:46:23 INFO mlflow.tracking.fluent: Experiment with name 'EXTRATREES' does not exist. Creating a new experiment.
2026/04/21 01:48:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 1 | Balanced Acc: 0.56999 | LogLoss: 0.43167


KeyboardInterrupt: 

### Catboost Core

In [117]:
for df in [X, test_data]:
    df['mulching_used'] = df['mulching_used'].map({'Yes': 1, 'No': 0})

    # ── Soil type lookup tables ──────────────────────────────────────────────
    FC  = {'sandy':0.18,'sandy_loam':0.25,'loam':0.35,'clay_loam':0.40,'clay':0.45}
    WP  = {'sandy':0.08,'sandy_loam':0.12,'loam':0.18,'clay_loam':0.22,'clay':0.27}
    EFF = {'sandy':0.75,'sandy_loam':0.80,'loam':0.85,'clay_loam':0.88,'clay':0.90}

    df['_fc'] = df['soil_type'].map(FC).fillna(0.30)
    df['_wp'] = df['soil_type'].map(WP).fillna(0.15)

    # ── Irrigation type efficiency lookup ────────────────────────────────────
    IRR_EFF = {'drip':0.92,'sprinkler':0.78,'furrow':0.65,'flood':0.55}
    df['irrigation_efficiency_factor'] = df['irrigation_type'].map(IRR_EFF).fillna(0.70)

    # ── Crop coefficient Kc by growth stage ──────────────────────────────────
    KC = {'initial':0.40,'development':0.70,'mid_season':1.15,'late_season':0.75,'harvest':0.40}
    df['growth_stage_kc'] = df['crop_growth_stage'].map(KC).fillna(0.75)

    # ════════════════════════════════════════════════════════════════════════
    # 1. WATER BALANCE FEATURES
    # ════════════════════════════════════════════════════════════════════════
    df['effective_rainfall']         = df['rainfall_mm'] * df['soil_type'].map(EFF).fillna(0.82)
    df['effective_irr_delivered']    = df['previous_irrigation_mm'] * df['irrigation_efficiency_factor']
    df['total_water_input']          = df['effective_rainfall'] + df['effective_irr_delivered']
    df['net_water_deficit']          = df['total_water_input'] - df['soil_moisture'] * df['field_area_hectare'] * 10
    df['water_supply_ratio']         = df['total_water_input'] / (df['soil_moisture'] + 1e-5)
    df['rainfall_irrigation_gap']    = df['previous_irrigation_mm'] - df['rainfall_mm']
    df['water_per_hectare']          = df['previous_irrigation_mm'] / (df['field_area_hectare'] + 1e-5)

    # ════════════════════════════════════════════════════════════════════════
    # 2. EVAPOTRANSPIRATION FEATURES
    # ════════════════════════════════════════════════════════════════════════
    # Reference ET (Hargreaves simplified — uses sunlight as Ra proxy)
    df['eto_hargreaves']             = (0.0023 * (df['temperature_c'] + 17.8)
                                        * (df['temperature_c'] ** 0.5)
                                        * df['sunlight_hours'] * 0.16)
    df['etc_crop_adjusted']          = df['eto_hargreaves'] * df['growth_stage_kc']

    # Vapour Pressure Deficit (kPa) — strongest atmospheric irrigation trigger
    df['vpd_kpa']                    = (0.6108
                                        * np.exp(17.27 * df['temperature_c'] / (df['temperature_c'] + 237.3))
                                        * (1 - df['humidity'] / 100))

    df['rainfall_effectiveness']     = df['rainfall_mm'] / (df['etc_crop_adjusted'] + 1e-5)
    df['et_deficit']                 = df['etc_crop_adjusted'] - df['soil_moisture'] * 0.3  # root zone factor
    df['crop_water_demand_per_day']  = df['etc_crop_adjusted'] / (df['sunlight_hours'] + 1e-5)

    # ════════════════════════════════════════════════════════════════════════
    # 3. SOIL WATER STATE FEATURES
    # ════════════════════════════════════════════════════════════════════════
    df['soil_moisture_deficit']      = df['_fc'] - df['soil_moisture'] / 100   # normalise moisture to 0–1
    df['relative_water_content']     = ((df['soil_moisture'] / 100 - df['_wp'])
                                        / (df['_fc'] - df['_wp'] + 1e-5)).clip(0, 1)
    df['soil_water_holding_cap']     = df['_fc'] * 1000 * (1 + df['organic_carbon'] * 0.05)
    df['salinity_osmotic_stress']    = df['electrical_conductivity'] * 0.64
    df['organic_carbon_moisture_bonus'] = df['organic_carbon'] * 3.5 * df['field_area_hectare']
    df['ph_deviation']               = (df['soil_ph'] - 6.5).abs()            # deviation from optimal

    # ════════════════════════════════════════════════════════════════════════
    # 4. CROP STRESS FEATURES
    # ════════════════════════════════════════════════════════════════════════
    CRITICAL_STAGES = {'flowering', 'grain_fill', 'pollination', 'mid_season'}
    df['critical_stage_flag']        = df['crop_growth_stage'].isin(CRITICAL_STAGES).astype(int)
    df['mulch_moisture_retention']   = df['mulching_used'] * 0.25 * df['soil_moisture']
    df['mulch_x_soil_moisture']      = df['mulching_used'] * df['soil_moisture']

    # ════════════════════════════════════════════════════════════════════════
    # 5. ENVIRONMENTAL INTERACTION FEATURES
    # ════════════════════════════════════════════════════════════════════════
    df['heat_stress_index']          = (df['temperature_c']
                                        * (1 - df['humidity'] / 100)
                                        * df['sunlight_hours'])
    df['wind_evaporation_boost']     = df['wind_speed_kmh'] * df['vpd_kpa'] * 0.07
    df['solar_radiation_index']      = df['sunlight_hours'] * 0.0864 * (df['temperature_c'] / 20)
    df['dew_point_gap']              = df['temperature_c'] - (df['temperature_c'] - (100 - df['humidity']) / 5)
    df['temp_humidity_interaction']  = df['temperature_c'] * (1 - df['humidity'] / 100)
    df['wind_x_vpd']                 = df['wind_speed_kmh'] * df['vpd_kpa']

    # ════════════════════════════════════════════════════════════════════════
    # 6. MANAGEMENT INTERACTION FEATURES
    # ════════════════════════════════════════════════════════════════════════
    df['ec_irrigation_interaction']  = df['electrical_conductivity'] * df['irrigation_efficiency_factor']
    df['water_source_reliability']   = df['water_source'].map(
                                        {'borewell':1.0,'canal':0.70,'pond':0.60,'rainwater':0.40}
                                       ).fillna(0.60)

    # ════════════════════════════════════════════════════════════════════════
    # 7. SPATIAL / TEMPORAL FEATURES
    # ════════════════════════════════════════════════════════════════════════
    df['aridity_index']              = df['rainfall_mm'] / (df['eto_hargreaves'] * 30 + 1e-5)
    df['season_dryness_flag']        = df['season'].isin(['summer','dry','winter']).astype(int)
    df['crop_x_season']              = df['crop_type'].astype(str) + '_' + df['season'].astype(str)
    df['region_x_crop']              = df['region'].astype(str) + '_' + df['crop_type'].astype(str)
    df['season_temp_interaction']    = df['season_dryness_flag'] * df['temperature_c']

    # ── Drop internal helper columns ─────────────────────────────────────────
    df.drop(columns=['_fc', '_wp'], inplace=True)

In [118]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [119]:
EXP_NAME = "CATBOOST"
RUN_NAME = f"Catgbm_seed{config.SEED}_{RUN}"

skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

final_oof_preds = np.zeros((len(X), n_classes))
final_test_preds = np.zeros((len(test_data), n_classes))
feature_importance = []
bacc_scores = []
ll_scores = []

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")
    mlflow.log_params({
		"seed": config.SEED,
		"n_folds": config.N_FOLDS,
		"n_cols": X.shape[1],
		**config.CATBOOST_PARAMS
	})

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]
        X_test = test_data.copy()
        
        # ---------- Rank Data ----------
        #for col in num_cols:
            #sorted_train = sorted_train = np.sort(X_train[col].values)

            #X_train[f"{col}_rank"] = rankdata(X_train[col].values)

            #X_valid[f"{col}_rank"] = np.searchsorted(sorted_train, X_valid[col].values)
            #X_test[f"{col}_rank"]  = np.searchsorted(sorted_train, X_test[col].values)
        
		# ---------- qcut ----------
        #for col in num_cols:
            #_, bins = pd.qcut(X_train[col], q=5, retbins=True, duplicates="drop")

            #X_train[f"{col}_bin"] = pd.cut(X_train[col], bins=bins, labels=False)
            #X_valid[f"{col}_bin"] = pd.cut(X_valid[col], bins=bins, labels=False)
            #X_test[f"{col}_bin"]  = pd.cut(X_test[col],  bins=bins, labels=False)

		# ------ Pipeline -----
        train_pool = cb.Pool(data=X_train, label=y_train, cat_features=cat_cols, thread_count=-1)
        valid_pool = cb.Pool(data=X_valid, label=y_valid, cat_features=cat_cols, thread_count=-1)
        test_pool = cb.Pool(data=X_test, cat_features=cat_cols, thread_count=-1)

        model = cb.train(
            train_pool,
            config.CATBOOST_PARAMS,
            eval_set=valid_pool
		)

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
            mlflow.catboost.log_model(model, name="model")

            oof_preds = model.predict(valid_pool, prediction_type='Probability')
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict(test_pool, prediction_type='Probability')
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=1,
                n_jobs=-1,
                random_state=config.SEED
            )
            feature_importance.append(fi.importances_mean)

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
        "recall_0": recall_per_class[0],
        "recall_1": recall_per_class[1],
        "recall_2": recall_per_class[2]
    }

    mlflow.log_metrics(metrics)

    # -------- FEATURE IMPORTANCE --------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")
    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)
    
    if fold < 5:
        del train_pool, valid_pool, test_pool, model, X_train, X_valid
        gc.collect()

# -------- SAVE FILES --------
oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)
save_csv_file(oof_df, os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv"))

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)
save_csv_file(test_df, os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv"))

Fold 1 | Balanced Acc: 0.96702 | LogLoss: 0.07792
Fold 2 | Balanced Acc: 0.96900 | LogLoss: 0.08059
Fold 3 | Balanced Acc: 0.96984 | LogLoss: 0.08110
Fold 4 | Balanced Acc: 0.96763 | LogLoss: 0.08126
Fold 5 | Balanced Acc: 0.96855 | LogLoss: 0.07701
Saving file to: artifacts\oof\Catgbm_seed42_v5_oof_proba.csv
Saving file to: artifacts\test_proba\Catgbm_seed42_v5_test_proba.csv


###  XGBoost TD

In [29]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [217]:
EXP_NAME = "XGBOOST_TD"
RUN_NAME = f"xgbm_TD_seed{config.SEED}_{RUN}"

skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

final_oof_preds = np.zeros((len(X), n_classes))
final_test_preds = np.zeros((len(test_data), n_classes))
feature_importance = []
bacc_scores = []
ll_scores = []

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")
    mlflow.log_params({
		"seed": config.SEED,
		"n_folds": config.N_FOLDS,
		"n_cols": X.shape[1],
		**config.XGB_TD_PARAMS
	})

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
        X_valid, y_valid = X.iloc[valid_idx].copy(), y.iloc[valid_idx]

		# ------ Pipeline -----

        model = XGB_TD_Classifier(**config.XGB_TD_PARAMS)
        model.fit(X_train, y_train, X_valid, y_valid, cat_col_names=cat_cols)

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
            mlflow.sklearn.log_model(model, name="model")

            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring='balanced_accuracy',
                n_repeats=1,
                n_jobs=-1,
                random_state=config.SEED
            )
            feature_importance.append(fi.importances_mean)

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
        "recall_0": recall_per_class[0],
        "recall_1": recall_per_class[1],
        "recall_2": recall_per_class[2]
    }

    mlflow.log_metrics(metrics)

    # -------- FEATURE IMPORTANCE --------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")
    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# -------- SAVE FILES --------
oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)
save_csv_file(oof_df, os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv"))

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)
save_csv_file(test_df, os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv"))

2026/04/18 01:14:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 01:14:13 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 1 | Balanced Acc: 0.96252 | LogLoss: 0.06047


2026/04/18 01:14:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 01:14:42 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 2 | Balanced Acc: 0.96417 | LogLoss: 0.05841


2026/04/18 01:15:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 01:15:30 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


...

Fold 4 | Balanced Acc: 0.96333 | LogLoss: 0.05688


2026/04/18 01:16:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 01:16:45 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 5 | Balanced Acc: 0.96309 | LogLoss: 0.05687


c:\Users\aryan\miniconda3\envs\kaggle\Lib\site-packages\sklearn\metrics\_classification.py:310: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Saving file to: artifacts\oof\xgbm_TD_seed42_v4_oof_proba.csv
Saving file to: artifacts\test_proba\xgbm_TD_seed42_v4_test_proba.csv
